In [ ]:
from google.colab import userdata

GROQ_API_KEY = userdata.get("GroqAPIkey")

In [ ]:
import os
os.environ["LANGSMITH_TRACING"] = "false"

In [ ]:
!cat .env

In [ ]:
!pip install -q \
langchain \
langchain-community \
langchain-text-splitters \
langchain-groq \
faiss-cpu \
pypdf \
python-pptx \
unstructured \
sentence-transformers \
langchain-chroma

In [ ]:
!pip install -U langchain-huggingface

In [ ]:
!mkdir documents

In [ ]:
import os
from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    UnstructuredPowerPointLoader
)

documents = []

folder_path = "documents"

for filename in os.listdir(folder_path):

    file_path = os.path.join(folder_path, filename)

    try:

        # PDF
        if filename.endswith(".pdf"):
            loader = PyPDFLoader(file_path)

        # TXT
        elif filename.endswith(".txt"):
            loader = TextLoader(file_path)

        # PPTX
        elif filename.endswith(".pptx"):
            loader = UnstructuredPowerPointLoader(file_path)

        else:
            continue

        docs = loader.load()

        # Add metadata
        for doc in docs:
            doc.metadata["source_file"] = filename
            doc.metadata["file_type"] = filename.split(".")[-1]

        documents.extend(docs)

    except Exception as e:
        print(f"Error loading {filename}: {e}")

print(f"Loaded {len(documents)} documents")

In [ ]:
import os
print ("FILES FOUND: ")
for filename in os.listdir("documents"):
    print (filename)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

split_docs = text_splitter.split_documents(documents)

print(f"Created {len(split_docs)} chunks")

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("Model downloaded succesfully")

In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    split_docs,
    embeddings
)

print("Vector database created successfully!")

In [ ]:
vectorstore.save_local("faiss_index")

In [ ]:
from langchain_community.vectorstores import FAISS

db = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 10
    }
)

In [ ]:

import os, json, textwrap
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv(override=True)
os.environ["LANGSMITH_TRACING"] = "false"

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
llm = ChatGroq(api_key=GROQ_API_KEY, model="llama-3.1-8b-instant")

In [ ]:


def extract_json_array(text: str) -> list:
    text = text.strip()
    if text.startswith("```"):
        lines = text.splitlines()
        text = "\n".join(lines[1:-1] if lines[-1].strip() == "```" else lines[1:])
    start, end = text.find("["), text.rfind("]")
    if start == -1 or end == -1:
        raise ValueError("No JSON array found in LLM response.")
    return json.loads(text[start:end+1])


def validate_questions(questions: list) -> list:
    required_keys = {"question", "options", "answer", "explanation"}  # added
    required_opts = {"A", "B", "C", "D"}
    valid = []
    for idx, q in enumerate(questions):
        if not required_keys.issubset(q.keys()):
            raise ValueError(f"Q{idx+1} missing keys: {required_keys - set(q.keys())}")
        if set(q["options"].keys()) != required_opts:
            raise ValueError(f"Q{idx+1} options must be exactly A, B, C, D")
        answer = str(q["answer"]).strip().upper()
        if answer not in required_opts:
            raise ValueError(f"Q{idx+1} answer '{answer}' not in A/B/C/D")
        if not str(q["explanation"]).strip():
            raise ValueError(f"Q{idx+1} explanation is empty")
        q["answer"] = answer
        valid.append(q)
    return valid


def verify_answer(llm, question: dict) -> tuple[str, str]:
    """Returns (verified_letter, explanation_for_correct_answer)."""
    opts_text = "\n".join(f"{k}. {v}" for k, v in question["options"].items())
    verify_prompt = f"""You are a precise answer checker.

Question: {question['question']}

Options:
{opts_text}

Task:
1. Identify the ONE factually correct answer letter (A, B, C, or D).
2. Write a concise 1-2 sentence explanation of WHY that option is correct
   and why the others are wrong.

Reply with ONLY valid JSON — no extra text:
{{"answer": "X", "explanation": "..."}}
"""
    for _ in range(3):
        try:
            raw = llm.invoke(verify_prompt).content.strip()
            if raw.startswith("```"):
                lines = raw.splitlines()
                raw = "\n".join(lines[1:-1] if lines[-1].strip() == "```" else lines[1:])
            start, end = raw.find("{"), raw.rfind("}") + 1
            obj = json.loads(raw[start:end])
            v = str(obj["answer"]).strip().upper()
            e = str(obj.get("explanation", "")).strip()
            if v in {"A", "B", "C", "D"} and e:
                return v, e
        except Exception:
            pass
    return question["answer"], question.get("explanation", "")


def verify_all_answers(llm, questions: list) -> list:
    print("🔎 Verifying answer keys and explanations...")
    corrections = 0
    for i, q in enumerate(questions):
        original = q["answer"]
        verified, explanation = verify_answer(llm, q)
        if verified != original:
            print(f"   ⚡ Q{i+1}: corrected {original} → {verified}")
            q["answer"] = verified
            corrections += 1
        q["explanation"] = explanation   # always overwrite with verified explanation
    if corrections == 0:
        print("✔ All answers verified — no corrections needed.")
    else:
        print(f"✔ Verified. {corrections} answer(s) corrected.")
    return questions


def generate_questions(llm, context: str, topic: str, num_questions: int, max_retries=3) -> list:
    prompt = f"""You are an AI teacher creating a multiple-choice quiz.

Generate EXACTLY {num_questions} MCQ questions about: {topic}

═══ QUESTION DESIGN RULES ═══

Write questions that have EXACTLY one defensible correct answer:

1. ASK FOR SPECIFICS, NOT GENERALITIES
   ❌ BAD:  "Which operator is used to combine conditions?"
   ✅ GOOD: "Which operator returns true ONLY when ALL conditions are true?"

2. USE "WHICH OF THE FOLLOWING IS NOT / INCORRECT"
   ✅ GOOD: "Which of the following is NOT a valid loop type in C++?"

3. PIN TO SPECIFIC BEHAVIOUR OR OUTPUT
   ✅ GOOD: "What is the output of: for(int i=0; i<3; i++) cout << i;"

4. USE ABSOLUTE LANGUAGE
   Use: "ONLY", "ALWAYS", "NEVER", "MUST", "EXCLUSIVELY"

═══ OPTION RULES ═══

- Exactly ONE option must be correct — re-read all 4 before finalising
- The other 3 must be WRONG — not partially correct, not "also valid in some cases"
- No two options may convey the same meaning, even with different words
- Do NOT use: "Both A and B", "All of the above", "None of the above"

═══ EXPLANATION RULES ═══

- Every question MUST include an "explanation" field
- 2-3 sentences: state why the correct answer is right AND why the key distractors are wrong
- Must be factually accurate and educational
- Written for a student who just got the question wrong

═══ SELF-CHECK ═══

For every question ask:
  - "Could a knowledgeable person defend MORE than one option?" → rewrite if yes
  - "Are any two options saying the same thing?" → replace one if yes

═══ OUTPUT FORMAT ═══

Return ONLY a valid JSON array. No markdown, no explanation outside JSON.

[
  {{
    "question": "...",
    "options": {{"A": "...", "B": "...", "C": "...", "D": "..."}},
    "answer": "X",
    "explanation": "The correct answer is X because ... The other options are wrong because ..."
  }}
]

CONTEXT:
{context}
"""
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            raw = extract_json_array(llm.invoke(prompt).content)
            if len(raw) != num_questions:
                raise ValueError(f"Expected {num_questions}, got {len(raw)}")
            return validate_questions(raw)
        except Exception as e:
            last_error = e
            print(f"⚠️ Attempt {attempt}/{max_retries} failed: {e}")
    raise RuntimeError(f"Generation failed after {max_retries} attempts: {last_error}")


def calculate_results(questions: list, user_answers: list) -> dict:
    score, weak = 0, []
    for i, (q, ua) in enumerate(zip(questions, user_answers)):
        if ua == q["answer"]:
            score += 1
        else:
            weak.append({
                "q_num":          i + 1,
                "question":       q["question"],
                "your_answer":    f"{ua}. {q['options'].get(ua, 'N/A')}",
                "correct_answer": f"{q['answer']}. {q['options'][q['answer']]}",
                "explanation":    q["explanation"],       # added
            })
    total = len(questions)
    return {
        "score":          score,
        "total":          total,
        "percentage":     round(score / total * 100, 1) if total else 0.0,
        "weak_questions": weak,
    }

In [ ]:

topic         = input("Enter topic: ").strip()
num_questions = int(input("How many MCQs? (1-20): ").strip())

print(f"\n🔍 Retrieving context for '{topic}'...")
docs    = retriever.invoke(topic)
context = "\n\n".join(d.page_content for d in docs)

if not context.strip():
    print("⚠️ No documents found for this topic.")
else:
    print(f"🤖 Generating {num_questions} questions...\n")
    questions = generate_questions(llm, context, topic, num_questions)
    questions = verify_all_answers(llm, questions)

    user_answers = []
    print("\n================ EXAM START ================\n")

    for i, q in enumerate(questions):
        print(f"Q{i+1}: {q['question']}\n")
        for letter, text in q["options"].items():
            print(f"  {letter}. {text}")

        while True:
            ans = input("\n  Your answer (A/B/C/D): ").strip().upper()
            if ans in {"A", "B", "C", "D"}:
                break
            print("  ⚠️ Invalid input. Enter A, B, C, or D.")

        user_answers.append(ans)
        correct = q["answer"]

        if ans == correct:
            print("  ✔ Correct!\n")
        else:
            print(f"  ✘ Wrong. Correct answer: {correct}. {q['options'][correct]}")
            print(f"\n  📖 Explanation: {q['explanation']}\n")   # shown immediately

        print("  " + "-" * 40)

    # ── results ───────────────────────────────
    results = calculate_results(questions, user_answers)

    grade_map = [
        (90, "Outstanding 🏆"),
        (75, "Great work 🎉"),
        (50, "Good effort 👍"),
        (0,  "Needs improvement 📚"),
    ]
    grade = next(label for threshold, label in grade_map if results["percentage"] >= threshold)

    print("\n================ RESULT ================\n")
    print(f"  Score   : {results['score']} / {results['total']}")
    print(f"  Percent : {results['percentage']}%")
    print(f"  Grade   : {grade}")

    if results["weak_questions"]:
        print("\n  ---- Topics to Revise ----\n")
        for wq in results["weak_questions"]:
            print(f"  Q{wq['q_num']}: {wq['question']}")
            print(f"       Your answer   : {wq['your_answer']}")
            print(f"       Correct answer: {wq['correct_answer']}")
            print(f"       📖 Explanation : {wq['explanation']}")
            print()
    else:
        print("\n  🎉 Excellent! No topics need revision.")

    print("\n========================================\n")

In [ ]:


def extract_json_array(text: str) -> list:
    text = text.strip()
    if text.startswith("```"):
        lines = text.splitlines()
        text = "\n".join(lines[1:-1] if lines[-1].strip() == "```" else lines[1:])
    start, end = text.find("["), text.rfind("]")
    if start == -1 or end == -1:
        raise ValueError("No JSON array found in LLM response.")
    return json.loads(text[start:end+1])


def validate_questions(questions: list) -> list:
    required_keys = {"question", "options", "answer", "explanation"}
    required_opts = {"A", "B", "C", "D"}
    valid = []
    for idx, q in enumerate(questions):
        if not required_keys.issubset(q.keys()):
            raise ValueError(f"Q{idx+1} missing keys: {required_keys - set(q.keys())}")
        if set(q["options"].keys()) != required_opts:
            raise ValueError(f"Q{idx+1} options must be exactly A, B, C, D")
        answer = str(q["answer"]).strip().upper()
        if answer not in required_opts:
            raise ValueError(f"Q{idx+1} answer '{answer}' not in A/B/C/D")
        if not str(q["explanation"]).strip():
            raise ValueError(f"Q{idx+1} explanation is empty")
        q["answer"] = answer
        valid.append(q)
    return valid


def verify_answer(llm, question: dict) -> tuple:
    opts_text = "\n".join(f"{k}. {v}" for k, v in question["options"].items())
    verify_prompt = f"""You are a precise answer checker.

Question: {question['question']}

Options:
{opts_text}

Task:
1. Identify the ONE factually correct answer letter (A, B, C, or D).
2. Write a concise 1-2 sentence explanation of WHY that option is correct
   and why the others are wrong.

Reply with ONLY valid JSON — no extra text:
{{"answer": "X", "explanation": "..."}}
"""
    for _ in range(3):
        try:
            raw = llm.invoke(verify_prompt).content.strip()
            if raw.startswith("```"):
                lines = raw.splitlines()
                raw = "\n".join(lines[1:-1] if lines[-1].strip() == "```" else lines[1:])
            start, end = raw.find("{"), raw.rfind("}") + 1
            obj = json.loads(raw[start:end])
            v = str(obj["answer"]).strip().upper()
            e = str(obj.get("explanation", "")).strip()
            if v in {"A", "B", "C", "D"} and e:
                return v, e
        except Exception:
            pass
    return question["answer"], question.get("explanation", "")


def verify_all_answers(llm, questions: list) -> list:
    print("🔎 Verifying answer keys and explanations...")
    corrections = 0
    for i, q in enumerate(questions):
        original = q["answer"]
        verified, explanation = verify_answer(llm, q)
        if verified != original:
            print(f"   ⚡ Q{i+1}: corrected {original} → {verified}")
            q["answer"] = verified
            corrections += 1
        q["explanation"] = explanation
    if corrections == 0:
        print("✔ All answers verified — no corrections needed.")
    else:
        print(f"✔ Verified. {corrections} answer(s) corrected.")
    return questions


def generate_questions(llm, context: str, topic: str, num_questions: int, max_retries=3) -> list:
    prompt = f"""You are an AI teacher creating a multiple-choice quiz.

Generate EXACTLY {num_questions} MCQ questions about: {topic}

═══ QUESTION DESIGN RULES ═══

Write questions that have EXACTLY one defensible correct answer:

1. ASK FOR SPECIFICS, NOT GENERALITIES
   ❌ BAD:  "Which operator is used to combine conditions?"
   ✅ GOOD: "Which operator returns true ONLY when ALL conditions are true?"

2. USE "WHICH OF THE FOLLOWING IS NOT / INCORRECT"
   ✅ GOOD: "Which of the following is NOT a valid loop type in C++?"

3. PIN TO SPECIFIC BEHAVIOUR OR OUTPUT
   ✅ GOOD: "What is the output of: for(int i=0; i<3; i++) cout << i;"

4. USE ABSOLUTE LANGUAGE
   Use: "ONLY", "ALWAYS", "NEVER", "MUST", "EXCLUSIVELY"

═══ OPTION RULES ═══

- Exactly ONE option must be correct — re-read all 4 before finalising
- The other 3 must be WRONG — not partially correct, not "also valid in some cases"
- No two options may convey the same meaning, even with different words
- Do NOT use: "Both A and B", "All of the above", "None of the above"

═══ EXPLANATION RULES ═══

- Every question MUST include an "explanation" field
- 2-3 sentences: why the correct answer is right AND why key distractors are wrong
- Must be factually accurate and educational

═══ SELF-CHECK ═══

For every question ask:
  - "Could a knowledgeable person defend MORE than one option?" → rewrite if yes
  - "Are any two options saying the same thing?" → replace one if yes

═══ OUTPUT FORMAT ═══

Return ONLY a valid JSON array. No markdown, no explanation outside JSON.

[
  {{
    "question": "...",
    "options": {{"A": "...", "B": "...", "C": "...", "D": "..."}},
    "answer": "X",
    "explanation": "The correct answer is X because ... The other options are wrong because ..."
  }}
]

CONTEXT:
{context}
"""
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            raw = extract_json_array(llm.invoke(prompt).content)
            if len(raw) != num_questions:
                raise ValueError(f"Expected {num_questions}, got {len(raw)}")
            return validate_questions(raw)
        except Exception as e:
            last_error = e
            print(f"⚠️ Attempt {attempt}/{max_retries} failed: {e}")
    raise RuntimeError(f"Generation failed after {max_retries} attempts: {last_error}")

In [ ]:
# ─────────────────────────────────────────────
# CELL 2 — Imports & setup
# ─────────────────────────────────────────────
import os, json, time, threading
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv(override=True)
os.environ["LANGSMITH_TRACING"] = "false"

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
llm = ChatGroq(api_key=GROQ_API_KEY, model="llama-3.1-8b-instant")

In [ ]:
# ─────────────────────────────────────────────
# CELL 4 — Mock test engine
# ─────────────────────────────────────────────

# ── Step 1: discover available topics from vectorstore ──

def get_available_topics(retriever, sample_queries: list[str]) -> list[str]:
    """
    Probes the vectorstore with broad queries to surface
    distinct topic labels stored in document metadata.
    Falls back to extracting topics from content if no metadata.
    """
    seen, topics = set(), []
    for query in sample_queries:
        docs = retriever.invoke(query)
        for doc in docs:
            # prefer explicit metadata topic tag
            topic = doc.metadata.get("topic") or doc.metadata.get("chapter") or \
                    doc.metadata.get("section") or doc.metadata.get("source")
            if topic and topic not in seen:
                seen.add(topic)
                topics.append(str(topic))
    return sorted(topics)

def generate_mock_questions(llm, context: str, topic: str, num_questions: int, max_retries=3) -> list:
    prompt = f"""You are an AI teacher creating a timed mock exam paper.

Generate EXACTLY {num_questions} MCQ questions on the topic: {topic}

═══ EXAM CONTEXT ═══

This is a TIMED MOCK TEST. Questions must:
- Cover a RANGE of difficulty: ~30% easy, ~50% medium, ~20% hard
- Test specific facts, behaviour, and application — not vague recall
- Be answerable in under 90 seconds each

═══ QUESTION DESIGN RULES ═══

Write questions with EXACTLY one defensible correct answer:

1. ASK FOR SPECIFICS, NOT GENERALITIES
   ❌ BAD:  "Which operator combines conditions?"
   ✅ GOOD: "Which operator returns true ONLY when ALL conditions are true?"

2. PIN TO SPECIFIC BEHAVIOUR OR OUTPUT
   ✅ GOOD: "What is the output of: for(int i=0; i<3; i++) cout << i;"
   ✅ GOOD: "What is the value of x after: int x=2; x = x << 1;"

3. USE ABSOLUTE LANGUAGE TO REMOVE AMBIGUITY
   Use: "ONLY", "ALWAYS", "NEVER", "MUST", "EXCLUSIVELY"

4. MIX QUESTION STYLES across the {num_questions} questions:
   - Direct fact: "What does X do?"
   - Predict output: "What is printed by...?"
   - Spot the error: "Which of the following is INCORRECT about X?"
   - Application: "Which approach would you use to...?"

═══ OPTION RULES ═══

- Exactly ONE option must be correct — re-read all 4 before finalising
- The other 3 must be WRONG — not partially correct, not "also valid"
- No two options may convey the same meaning even with different words
- Do NOT use: "Both A and B", "All of the above", "None of the above"
- Rotate where the correct answer appears — do not always use "B" or "C"

═══ EXPLANATION RULES ═══

- Every question MUST include an "explanation" field
- 2-3 sentences: why the correct answer is right AND why the key distractors are wrong
- Pitched at a student who just got it wrong in an exam

═══ SELF-CHECK (run before returning) ═══

For every question confirm:
  ✔ Only one option is defensibly correct
  ✔ No two options say the same thing
  ✔ The explanation matches the nominated answer
  ✔ Difficulty is mixed across the question set

═══ OUTPUT FORMAT ═══

Return ONLY a valid JSON array. No markdown, no text outside the array.

[
  {{
    "question": "...",
    "difficulty": "easy" | "medium" | "hard",
    "options": {{"A": "...", "B": "...", "C": "...", "D": "..."}},
    "answer": "X",
    "explanation": "The correct answer is X because ... The other options are wrong because ..."
  }}
]

CONTEXT:
{context}
"""
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            raw = extract_json_array(llm.invoke(prompt).content)
            if len(raw) != num_questions:
                raise ValueError(f"Expected {num_questions}, got {len(raw)}")
            # inject difficulty default if LLM omits it
            for q in raw:
                q.setdefault("difficulty", "medium")
            return validate_questions(raw)
        except Exception as e:
            last_error = e
            print(f"  ⚠️ Attempt {attempt}/{max_retries} failed: {e}")
    raise RuntimeError(f"Generation failed after {max_retries} attempts: {last_error}")


# ── Step 2: timer (runs in background thread) ────────────

class MockTimer:
    """
    Counts down in a background thread.
    Sets self.expired = True when time runs out.
    Call .start() to begin, .stop() to cancel early.
    """
    def __init__(self, duration_seconds: int):
        self.duration  = duration_seconds
        self.remaining = duration_seconds
        self.expired   = False
        self._stop_evt = threading.Event()
        self._thread   = threading.Thread(target=self._run, daemon=True)

    def _run(self):
        while self.remaining > 0 and not self._stop_evt.is_set():
            time.sleep(1)
            self.remaining -= 1
        if not self._stop_evt.is_set():
            self.expired = True

    def start(self):
        self._thread.start()

    def stop(self):
        self._stop_evt.set()

    def elapsed(self) -> int:
        return self.duration - self.remaining

    def fmt_remaining(self) -> str:
        m, s = divmod(self.remaining, 60)
        return f"{m:02d}:{s:02d}"


# ── Step 3: report card (all calculation in Python) ──────

def build_report_card(
    all_questions:  list,
    user_answers:   dict,        # {flat_index: "A"/"B"/"C"/"D"/None}
    topic_map:      dict,        # {flat_index: topic_name}
    time_taken_sec: int,
    total_sec:      int,
) -> dict:
    """
    Computes every metric in pure Python — no LLM involved.
    Returns a structured report dict.
    """
    total      = len(all_questions)
    correct    = 0
    wrong      = 0
    unanswered = 0
    weak       = []                     # questions to revise
    topic_stats: dict[str, dict] = {}   # per-topic breakdown

    for idx, q in enumerate(all_questions):
        topic  = topic_map[idx]
        ua     = user_answers.get(idx)          # None = unanswered

        # initialise topic bucket
        if topic not in topic_stats:
            topic_stats[topic] = {"correct": 0, "wrong": 0, "unanswered": 0, "total": 0}
        topic_stats[topic]["total"] += 1

        if ua is None:
            unanswered += 1
            topic_stats[topic]["unanswered"] += 1
            weak.append({
                "q_num":          idx + 1,
                "topic":          topic,
                "question":       q["question"],
                "your_answer":    "Not answered",
                "correct_answer": f"{q['answer']}. {q['options'][q['answer']]}",
                "explanation":    q["explanation"],
                "status":         "unanswered",
            })
        elif ua == q["answer"]:
            correct += 1
            topic_stats[topic]["correct"] += 1
        else:
            wrong += 1
            topic_stats[topic]["wrong"] += 1
            weak.append({
                "q_num":          idx + 1,
                "topic":          topic,
                "question":       q["question"],
                "your_answer":    f"{ua}. {q['options'].get(ua, 'N/A')}",
                "correct_answer": f"{q['answer']}. {q['options'][q['answer']]}",
                "explanation":    q["explanation"],
                "status":         "wrong",
            })

    percentage = round(correct / total * 100, 1) if total else 0.0

    # topics where accuracy < 60%
    revise_topics = [
        t for t, s in topic_stats.items()
        if s["total"] > 0 and (s["correct"] / s["total"]) < 0.6
    ]

    grade_map = [
        (90, "Outstanding 🏆"),
        (75, "Great work 🎉"),
        (50, "Good effort 👍"),
        (0,  "Needs improvement 📚"),
    ]
    grade = next(label for threshold, label in grade_map if percentage >= threshold)

    m, s   = divmod(time_taken_sec, 60)
    tm, ts = divmod(total_sec, 60)

    return {
        "total":        total,
        "correct":      correct,
        "wrong":        wrong,
        "unanswered":   unanswered,
        "percentage":   percentage,
        "grade":        grade,
        "time_taken":   f"{m:02d}:{s:02d}",
        "time_allowed": f"{tm:02d}:{ts:02d}",
        "topic_stats":  topic_stats,
        "revise_topics":revise_topics,
        "weak":         weak,
    }


def print_report_card(report: dict):
    SEP = "=" * 52

    print(f"\n{SEP}")
    print("            📋  MOCK TEST REPORT CARD")
    print(SEP)
    print(f"  Score      : {report['correct']} / {report['total']}")
    print(f"  Correct    : {report['correct']}")
    print(f"  Wrong      : {report['wrong']}")
    print(f"  Unanswered : {report['unanswered']}")
    print(f"  Percentage : {report['percentage']}%")
    print(f"  Grade      : {report['grade']}")
    print(f"  Time taken : {report['time_taken']} / {report['time_allowed']}")

    print(f"\n  ── Per-topic breakdown ──")
    for topic, s in report["topic_stats"].items():
        pct = round(s["correct"] / s["total"] * 100) if s["total"] else 0
        bar = ("█" * (pct // 10)).ljust(10)
        print(f"  {topic:<28} [{bar}] {pct}%  "
              f"({s['correct']}✔  {s['wrong']}✘  {s['unanswered']}–)")

    # difficulty breakdown (calculated in build_report_card)
    if report.get("difficulty_stats"):
        print(f"\n  ── Difficulty breakdown ──")
        for diff in ["easy", "medium", "hard"]:
            s = report["difficulty_stats"].get(diff, {"correct": 0, "total": 0})
            if s["total"] == 0:
                continue
            pct = round(s["correct"] / s["total"] * 100)
            print(f"  {diff:<8} : {s['correct']}/{s['total']}  ({pct}%)")

    if report["revise_topics"]:
        print(f"\n  ── Topics to revise (< 60% accuracy) ──")
        for t in report["revise_topics"]:
            print(f"   ⚠  {t}")
    else:
        print("\n  ✅ Strong performance across all topics!")

    if report["weak"]:
        print(f"\n{SEP}")
        print("       📖  WRONG / UNANSWERED — WITH EXPLANATIONS")
        print(SEP)
        for wq in report["weak"]:
            status = "✘ Wrong" if wq["status"] == "wrong" else "– Unanswered"
            diff   = wq.get("difficulty", "medium")
            print(f"\n  Q{wq['q_num']} [{wq['topic']}]  [{diff}]  {status}")
            print(f"  {wq['question']}")
            print(f"  Your answer   : {wq['your_answer']}")
            print(f"  Correct answer: {wq['correct_answer']}")
            print(f"  📖 {wq['explanation']}")
            print("  " + "-" * 48)

    print(f"\n{SEP}\n")


# ── Step 4: full mock test flow ───────────────────────────

def run_mock_test(llm, retriever):

    SEP = "─" * 52
    print("\n" + "=" * 52)
    print("           🎓  MOCK TEST  —  SETUP")
    print("=" * 52)

    # ── discover topics ───────────────────────────────────
    print("\n🔍 Scanning available topics...\n")
    probe_queries = [
        "introduction", "basics", "functions", "loops",
        "arrays", "pointers", "classes", "data structures",
        "sorting", "recursion", "memory", "operators"
    ]
    topics = get_available_topics(retriever, probe_queries)

    if not topics:
        print("⚠️  Could not detect topics from vectorstore metadata.")
        print("   Enter topics manually, comma-separated:")
        raw = input("   Topics: ").strip()
        topics = [t.strip() for t in raw.split(",") if t.strip()]

    print("  Available topics:\n")
    for i, t in enumerate(topics, 1):
        print(f"    {i:>2}. {t}")

    # ── topic selection ───────────────────────────────────
    print(f"\n{SEP}")
    print("  Select topics by number (e.g. 1,3,5  or  'all'):")
    raw = input("  → ").strip()

    if raw.lower() == "all":
        chosen_topics = topics
    else:
        try:
            indices      = [int(x.strip()) - 1 for x in raw.split(",")]
            chosen_topics = [topics[i] for i in indices if 0 <= i < len(topics)]
        except ValueError:
            print("⚠️  Invalid selection. Using all topics.")
            chosen_topics = topics

    if not chosen_topics:
        print("⚠️  No topics selected. Aborting.")
        return

    print(f"\n  ✔ Selected: {', '.join(chosen_topics)}")

    # ── questions per topic ───────────────────────────────
    print(f"\n{SEP}")
    while True:
        try:
            q_per_topic = int(input(
                f"  Questions per topic (1–10, {len(chosen_topics)} topic(s) selected): "
            ).strip())
            if 1 <= q_per_topic <= 10:
                break
            print("  ⚠️  Enter a number between 1 and 10.")
        except ValueError:
            print("  ⚠️  Please enter a valid integer.")

    total_questions = len(chosen_topics) * q_per_topic
    print(f"  Total questions: {total_questions}")

    # ── time duration ─────────────────────────────────────
    print(f"\n{SEP}")
    print("  Time limit options:")
    time_options = {
        "1": ("10 minutes",  600),
        "2": ("20 minutes", 1200),
        "3": ("30 minutes", 1800),
        "4": ("45 minutes", 2700),
        "5": ("60 minutes", 3600),
        "6": ("Custom",        -1),
    }
    for k, (label, _) in time_options.items():
        print(f"    {k}. {label}")

    while True:
        t_choice = input("  → ").strip()
        if t_choice in time_options:
            break
        print("  ⚠️  Enter 1–6.")

    if t_choice == "6":
        while True:
            try:
                duration_sec = int(input("  Enter duration in minutes: ").strip()) * 60
                if duration_sec > 0:
                    break
                print("  ⚠️  Must be > 0.")
            except ValueError:
                print("  ⚠️  Please enter a valid integer.")
    else:
        duration_sec = time_options[t_choice][1]

    m, s = divmod(duration_sec, 60)
    print(f"\n  ✔ Time allowed: {m:02d}:{s:02d}")

    # ── generate all questions ────────────────────────────
    print(f"\n{'=' * 52}")
    print("  🤖 Generating questions...\n")

    all_questions: list  = []
    topic_map:     dict  = {}   # flat_index → topic name

    for topic in chosen_topics:
        print(f"  [{topic}] retrieving context...")
        docs    = retriever.invoke(topic)
        context = "\n\n".join(d.page_content for d in docs)
        if not context.strip():
            print(f"  ⚠️  No documents found for '{topic}' — skipping.")
            continue

        print(f"  [{topic}] generating {q_per_topic} questions...")
        try:
            qs = generate_mock_questions(llm, context, topic, q_per_topic)
            qs = verify_all_answers(llm, qs)
        except RuntimeError as e:
            print(f"  ❌ {topic}: {e} — skipping.")
            continue

        for q in qs:
            topic_map[len(all_questions)] = topic
            all_questions.append(q)

    if not all_questions:
        print("\n❌ No questions generated. Aborting mock test.")
        return

    actual_total = len(all_questions)
    print(f"\n  ✔ {actual_total} questions ready across "
          f"{len(set(topic_map.values()))} topic(s).\n")

    input("  Press Enter to start the timer and begin the test...")

    # ── start timer ───────────────────────────────────────
    timer = MockTimer(duration_sec)
    timer.start()

    user_answers: dict = {}   # flat_index → letter

    print(f"\n{'=' * 52}")
    print("           ⏱  MOCK TEST  —  BEGIN")
    print(f"{'=' * 52}\n")
    print("  Type your answer (A/B/C/D) or:")
    print("    's' = skip question")
    print("    'q' = submit test early\n")
    print(f"{'─' * 52}\n")

    # ── question loop ─────────────────────────────────────
    for idx, q in enumerate(all_questions):

        # check timer before each question
        if timer.expired:
            print("\n  ⏰ Time's up! Submitting automatically...\n")
            break

        topic = topic_map[idx]
        remaining = timer.fmt_remaining()

        print(f"  Q{idx+1}/{actual_total}  [{topic}]  ⏱ {remaining}\n")
        print(f"  {q['question']}\n")
        for letter, text in q["options"].items():
            print(f"    {letter}. {text}")

        while True:
            if timer.expired:
                print("\n  ⏰ Time's up!\n")
                break

            ans = input("\n  Answer (A/B/C/D / s=skip / q=quit): ").strip().upper()

            if ans == "Q":
                timer.stop()
                print("\n  ✔ Test submitted early.\n")
                # fill remaining as unanswered
                for remaining_idx in range(idx, actual_total):
                    user_answers.setdefault(remaining_idx, None)
                break

            if ans == "S":
                user_answers[idx] = None
                print("  – Skipped.\n")
                break

            if ans in {"A", "B", "C", "D"}:
                user_answers[idx] = ans
                print()
                break

            print("  ⚠️  Enter A, B, C, D, 's', or 'q'.")

        print(f"  {'─' * 48}\n")

        if ans == "Q":
            break

    timer.stop()
    time_taken = timer.elapsed()

    # ensure every question has an entry (unanswered if not reached)
    for i in range(actual_total):
        user_answers.setdefault(i, None)

    # ── build and print report ────────────────────────────
    report = build_report_card(
        all_questions, user_answers, topic_map, time_taken, duration_sec
    )
    print_report_card(report)

In [ ]:
run_mock_test(llm, retriever)